# Prueba técnica Creceré AI — Etapa 2: transcripción completa y análisis

Etapa 1 (`etapa1.ipynb`) dejó el perfil técnico de los 100 audios y un benchmark A/B
(Deepgram vs ElevenLabs) sobre 6 casos. **ElevenLabs ganó** el benchmark: mayor confianza
media, menor tasa de confianza baja y mejor segmentación de turnos en los 6 casos
comparados. Es el motor que se usa de aquí en adelante.

## Qué hace este notebook

1. **Extracción completa** — transcribe los 100 audios (50 humanos / 50 IA) con
   ElevenLabs y deja un `.txt` por audio (turnos de hablante + `[CENSURADO]` en los
   huecos de pitido), igual que se hizo con los 6 casos del benchmark.
2. *(siguiente)* Análisis sobre las transcripciones completas.


---
## 1. Configuración

In [2]:
import json, os, sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))
from transcribe import elevenlabs, norm_elevenlabs, marcar_censura, por_turnos, CARPETAS

ROOT = Path("..")                             # carpeta que contiene las dos carpetas de audios
OUT = Path("salidas/transcripciones")         # entregable: un .txt por audio
CACHE = Path("cache/elevenlabs")              # cache de la respuesta cruda, fuera de salidas/
CENSURA_PATH = Path("salidas/censura.json")   # pitidos: calibrados y validados en etapa1.ipynb

EL_KEY = os.getenv("ELEVENLABS_API_KEY", "")

# Si el entorno no está tomando la key (el kernel se inició antes de exportarla), pégala
# aquí directo y corre esta celda de nuevo:
# EL_KEY = "sk_..."

print(f"ELEVENLABS_API_KEY: {'ok' if EL_KEY else 'FALTA'} (len={len(EL_KEY)})")

if not CENSURA_PATH.exists():
    raise FileNotFoundError(
        f"No existe {CENSURA_PATH}. Lo genera etapa1.ipynb, sección 2 (celda 2e67b3b8), "
        "después de calibrar (Fase 1) y detectar (Fase 2) los pitidos. Corre esa sección "
        "primero -- no uses data/eventos_censura.json, que sale de un script aparte "
        "(src/audio_profile.py) sin la calibración ni la validación por escucha de acá."
    )
censura = json.loads(CENSURA_PATH.read_text(encoding="utf-8"))
AUDIOS = [(g, p) for g, c in CARPETAS.items() for p in sorted((ROOT / c).glob("*.wav"))]
print(f"{len(AUDIOS)} audios:", {g: sum(1 for x, _ in AUDIOS if x == g) for g in CARPETAS})

ELEVENLABS_API_KEY: ok (len=51)
100 audios: {'humano': 50, 'ia': 50}


---
## 2. Extracción completa (ElevenLabs)

Misma lógica que el benchmark de Etapa 1 (`marcar_censura` + `por_turnos`, ahora en
`src/transcribe.py` para no duplicar código entre notebooks), aplicada a los 100 audios.

Los pitidos se toman de `salidas/censura.json`, el archivo que genera **el propio
`etapa1.ipynb`** en su sección 2, después de: (1) calibrar los umbrales escuchando 24
marcas a mano, (2) detectarlos en los 100 audios, y (3) validar una muestra de 20
detecciones escuchándolas. Esa es la trazabilidad que justifica el archivo, por eso se usa
ese y no `data/eventos_censura.json` -- ese otro sale de `src/audio_profile.py`, un script
aparte que usa los mismos umbrales pero corre por fuera del notebook, sin el proceso de
calibración/validación documentado acá.

Por cada audio: si ya existe su respuesta cacheada en `cache/elevenlabs/`, se reusa (no se
vuelve a pagar); si no, se llama a la API, se normaliza y se cachea. El `.txt` final se
regenera siempre a partir de las palabras (cacheadas o nuevas), en
`salidas/transcripciones/<grupo>/<archivo>.txt`.

In [5]:
import time

OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

def extraer_todo():
    if not EL_KEY:
        print("Falta ELEVENLABS_API_KEY en el entorno."); return
    for i, (grupo, path) in enumerate(AUDIOS, 1):
        stem = path.stem
        txt_path = OUT / grupo / f"{stem}.txt"
        cache_path = CACHE / f"{grupo}__{stem}.json"
        txt_path.parent.mkdir(parents=True, exist_ok=True)

        if cache_path.exists():
            words = json.loads(cache_path.read_text(encoding="utf-8"))
        else:
            t0 = time.time()
            try:
                j = elevenlabs(path, EL_KEY)
            except Exception as e:
                print(f"  [{i}/{len(AUDIOS)}] x {grupo}/{stem}: {e}")
                continue
            words = norm_elevenlabs(j)
            cache_path.write_text(json.dumps(words, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(AUDIOS)}] {grupo}/{stem}: {len(words)} palabras ({time.time()-t0:.0f}s)")

        texto = por_turnos(marcar_censura(words, censura.get(path.name, [])))
        txt_path.write_text(texto, encoding="utf-8")

extraer_todo()

  [94/100] ia/c1a958de-6df6-488a-8342-3cc44f4e31c6: 366 palabras (5s)
  [95/100] ia/c3d816dd-e0cc-46ef-b2db-a7d31a0bd587: 78 palabras (3s)
  [96/100] ia/c8b57e01-57b6-4d85-9613-c786aef9f8a0: 743 palabras (8s)
  [97/100] ia/da02116f-5f55-4cc7-a800-373f4939fa01: 835 palabras (10s)
  [98/100] ia/edbc4154-c6ce-4a77-b899-caf949e1815a: 98 palabras (3s)
  [99/100] ia/f55b661d-8a1b-4528-976e-d7b2f565b2ea: 1396 palabras (12s)
  [100/100] ia/fc9da4d5-10dd-4aab-afb0-de918f081149: 152 palabras (3s)


---
## 3. Verificación

In [6]:
hechos = sorted(OUT.glob("*/*.txt"))
print(f"{len(hechos)} transcripciones en {OUT}")
{g: sum(1 for p in hechos if p.parent.name == g) for g in CARPETAS}

100 transcripciones en salidas\transcripciones


{'humano': 50, 'ia': 50}

In [7]:
ejemplo = hechos[0]
print(ejemplo)
print(ejemplo.read_text(encoding="utf-8")[:1000])

salidas\transcripciones\humano\0445c357-e465-49ae-8067-bfa91d11b532.txt
[00:07] speaker_1: [CENSURADO]
[00:11] speaker_1: [CENSURADO]
[00:14] speaker_1: [CENSURADO]
[00:15] speaker_1: Aló.
[00:16] speaker_0: Aló, ¿hablo con la señora [CENSURADO]
[00:20] speaker_0: ?
[00:20] speaker_1: ¿Y quién la necesita?
[00:22] speaker_0: Eh, mire, mucho gusto. Mi nombre [CENSURADO] me comunico [CENSURADO] Nos comunicamos porque realizamos un acuerdo de pago con ella y para hoy tenemos la cuota estipulada. Queremos confirmar si contamos con este pago para el día de hoy.
[00:39] speaker_1: El día de hoy, no.
[00:41] speaker_0: ¿Cuándo podemos contar con el pago? La cuota está por... el acuerdo está por ochocientos mil pesos, dos cuotas, cada una por cuatrocientos mil pesos. Queremos confirmar que, pues, qué fecha dentro del mes puede realizar el pago.
[00:57] speaker_1: Sí, para el otro mes.
[01:00] speaker_0: Y esto nos puedes hacer uno a uno para mantener el descuento de cien mil pesos. Estos cien 